## Building a Chatbot
In this section we'll go over an example of how to design and implement an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.
Note that this chatbot that we build will only use the language model to have a conversation. There are several other related concepts that you may be looking for.
* Conversational RAG: Enable a chatbot experience over an external source of data.
* Agents: Build a chatbot that can take actions

This section tutorial will cover the basics which will be helpful for those two more advanced topics.


In [1]:
import os
from dotenv import load_dotenv
load_dotenv() # loading all the environment variables
groq_api_key=os.getenv("GROQ_API_KEY")

In [2]:
from langchain_groq import ChatGroq
model=ChatGroq(model='llama-3.3-70b-versatile',api_key=groq_api_key)
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.1', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x107e9c980>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x107e9d6a0>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [3]:
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content="Hi, My name is Gaurav and I am a software Engineer!")])

AIMessage(content="Nice to meet you, Gaurav! It's great to hear that you're a software engineer. What kind of projects do you usually work on, and what programming languages are you most familiar with?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 42, 'prompt_tokens': 50, 'total_tokens': 92, 'completion_time': 0.091657982, 'completion_tokens_details': None, 'prompt_time': 0.005661615, 'prompt_tokens_details': None, 'queue_time': 0.052368029, 'total_time': 0.097319597}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fc336-7a2f-7ac0-a940-76d92cf83192-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 50, 'output_tokens': 42, 'total_tokens': 92})

In [4]:
from langchain_core.messages import AIMessage
model.invoke(
    [
        HumanMessage(content="Hi, My name is Gaurav and I am a software Engineer!"),
        AIMessage(content="Nice to meet you, Gaurav! Welcome! It's great to hear that you're a software engineer. What kind of projects do you usually work on, or what technologies are you most interested in? I'm here to chat and help with any questions or topics you'd like to discuss!"),
        HumanMessage(content="Hey, what's my name and what my profession"),
    ]
)

AIMessage(content="Your name is Gaurav, and you're a Software Engineer.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 130, 'total_tokens': 145, 'completion_time': 0.026602375, 'completion_tokens_details': None, 'prompt_time': 0.012809457, 'prompt_tokens_details': None, 'queue_time': 0.051401442, 'total_time': 0.039411832}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fc336-9c93-7402-8ebe-5400ce02310e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 130, 'output_tokens': 15, 'total_tokens': 145})

### Message History
We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this.

In [5]:
from  langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import  BaseChatMessageHistory
from langchain_core.runnables import RunnableWithMessageHistory

store={}
def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]
with_message_history=RunnableWithMessageHistory(
    model,
    get_session_history
)


/var/folders/tw/n0t6m6cx57702f9c9ldw7z200000gn/T/ipykernel_5998/2876733093.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from  langchain_community.chat_message_histories import ChatMessageHistory
/Users/apple/WorkSpace/Agentic AI and Gen AI/.venv/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [6]:
config={
    "configurable":{
        "session_id":"chat1"
    }
}

In [7]:
response=with_message_history.invoke(
    [
        HumanMessage(content="Hi, My name is Gaurav and I am a software Engineer!"),

     ],
    config=config
)

In [8]:
response.content

"Hi Gaurav, nice to meet you! It's great to hear that you're a software engineer. What kind of projects do you usually work on, and what programming languages are you most proficient in?"

In [9]:
response=with_message_history.invoke(
    [
        HumanMessage(content="Hi, What is my name?"),

     ],
    config=config
)
response.content


"Your name is Gaurav, and you're a software engineer."

#### Prompt Template
Prompt Template help to turn raw user information into a formate that the LLM can work with. In this case the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First let's add in a system message with some custom instructions (but still taking message as input ). Next. we'll add in more input besides just the messages.


In [10]:
from langchain_core.prompts import  ChatPromptTemplate,MessagesPlaceholder
prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant. Answer all the question to the best of your ability"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain=prompt|model

In [11]:
chain.invoke(
    {
        "messages":[
            HumanMessage(content="Hi, My name is Gaurav and I am a software Engineer!"),
        ]
    }
)

AIMessage(content="Nice to meet you, Gaurav! It's great to hear that you're a software engineer. That's a fascinating field, and I'm sure you must be working on some exciting projects. What kind of projects do you usually work on, and what programming languages are you proficient in?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 60, 'prompt_tokens': 66, 'total_tokens': 126, 'completion_time': 0.16533302, 'completion_tokens_details': None, 'prompt_time': 0.002424551, 'prompt_tokens_details': None, 'queue_time': 0.055531313, 'total_time': 0.167757571}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fc336-c796-7e90-b643-d0ea8d90ab1d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 66, 'output_tokens': 60, 'total_tokens': 126})

In [12]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history)

/Users/apple/WorkSpace/Agentic AI and Gen AI/.venv/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [13]:
config={
    "configurable":{
        "session_id":"chat3"
    }
}

In [14]:
response=with_message_history.invoke(
    [
        HumanMessage(content="Hi, My name is Gaurav and I am a software Engineer!"),

     ],
    config=config
)

In [15]:
response.content

"Hello Gaurav, nice to meet you! It's great to know that you're a software engineer. What kind of projects do you usually work on, or what programming languages are you most interested in? I'm here to help with any questions or topics you'd like to discuss. How's your day going so far?"

In [16]:
response=with_message_history.invoke(
    [
        HumanMessage(content="Hi, what is my name?"),

     ],
    config=config
)

In [17]:
response.content

"Your name is Gaurav, and you're a software engineer. I remember!"

In [18]:
### Add more complexity
prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant. Answer all questions to the best of your ability in {language}"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain=prompt|model

In [19]:
response=chain.invoke({
    "messages":[
        HumanMessage(content="Hi, my name is Gaurav and I am a software Engineer!"),
    ],
    "language":"Hindi"
})
response.content

'नमस्ते गौरव जी, सॉफ्टवेयर इंजीनियर के रूप में आपका स्वागत है! मैं आपकी कैसे मदद कर सकता हूँ?'

Let's now wrap this more complicated chain in a Message History class. This time, because there are multiple keys in the input we need to specify the correct ket to use to save the chat history.

In [20]:
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)


/Users/apple/WorkSpace/Agentic AI and Gen AI/.venv/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [21]:
config={
    "configurable":{
        "session_id":"chat4"
    }
}

response=with_message_history.invoke(
    {
        "messages":[HumanMessage(content="Hi, my name is Gaurav and I am a software Engineer!")],
        "language":"Hindi"
    },
    config=config
)
response.content


'नमस्ते गौरव! मैं आपका सहायक हूँ। यह जानकर खुशी हुई कि आप एक सॉफ्टवेयर इंजीनियर हैं। सॉफ्टवेयर इंजीनियरिंग एक बहुत ही रोचक और चुनौतीपूर्ण क्षेत्र है। क्या आप अपने काम के बारे में कुछ बताना चाहेंगे या किसी विशिष्ट विषय पर चर्चा करना चाहेंगे?'

In [22]:
response=with_message_history.invoke(
    {
        "messages":[HumanMessage(content="Hi, What's my name?")],
        "language":"Hindi"
    },
    config=config
)
response.content


'आपका नाम गौरव है!'

#### Managing the Conversation History
One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context widow of the LLM, Therefore it is important to add a step that limits the size of the messages you are passing in.

'trim_message' helper to reduce how many message we're sending to the model. The trimmer allows us to specify how many tokens we want to keep along with other parameters like if we want to always keep the system message and whether to allow partial messages.

In [34]:
from langchain_core.messages import SystemMessage,trim_messages
from langchain_core.messages.utils import count_tokens_approximately

trimmer=trim_messages(
    max_tokens=70,
    strategy="last",
    token_counter=count_tokens_approximately,
    include_system=True,
    allow_partial=False,
    start_on="human"
)

messages=[
    SystemMessage(content="You are a good assistant"),
    HumanMessage(content="Hi I'm Gaurav"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2+2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="No problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!")
]

trimmer.invoke(messages)

[SystemMessage(content='You are a good assistant', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='whats 2+2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='No problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [41]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

chain=(
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
    |prompt
    |model
)

response=chain.invoke(
    {
        "messages":messages+[HumanMessage(content="which math problem asked")],
        "language":"Hindi"
    }


)

response.content

'आपके द्वारा जो गणित का प्रश्न पूछा गया था, वह था 2+2।'

In [37]:
## Lets wrap this in this Message History

with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

config={
    "configurable":{
        "session_id":"chat5"
    }
}

In [38]:
response=with_message_history.invoke(
    {
        "messages":messages+[HumanMessage(content="What is my name?")],
        "language":"Hindi"
    },
    config=config
)

response.content

'मुझे खेद है, लेकिन मैं आपका नाम नहीं जानता हूँ। हमारी बातचीत अभी शुरू हुई है, और आपने अपना नाम नहीं बताया है।'